# 04 — LSTM Training Code Explained (Flickr8k)

**Companion to:** `notebooks/03_lstm_training.ipynb` and `src/train_lstm.py`.

This notebook explains **why each step is required**, **what the code does**, and **what its inputs and outputs mean**. It follows the existing repository architecture diagram; it does **not** replace the working training code.

**Two modes:**
- **Demonstration** cells use a tiny synthetic dataset; no Flickr8k download or VGG16 feature extraction is needed.
- **Full training** is an optional last section that reuses your real `data/flickr8k` and `data/features` folders.

This notebook does not implement greedy decoding, beam search or BLEU evaluation. Those are later stages.

## 1. Image captioning in this project

**Goal:** Given an image and some previously generated words, predict the **next word**.

Your diagram has two parallel branches:

| Image branch | Caption branch |
|---|---|
| Flickr8k image resized to 224×224 | Five text captions per Flickr8k image |
| Frozen ImageNet VGG16 | Cleaning, start/end markers, token IDs, padding |
| Save a **4096-dimensional** vector from VGG16's `fc2` layer | **Embedding(256)** |
| Train a **Dense(256)** layer on the saved vector | Train **LSTM(256)** |

The two resulting **256-dimensional** vectors are **added** (feature fusion), then passed through **Dense(256) → Softmax(vocabulary)** to calculate next-word probabilities.

**Key distinction:** The VGG16 encoder is pretrained and frozen; the image Dense layer, word embedding, LSTM, fusion Dense layer, and output layer are learned during caption-decoder training. The image vector is fused with the *LSTM output*, not supplied directly as one of its recurrent time steps.

## 2. Imports and working directory

We import **existing repository functions** rather than copying them into this notebook. This avoids maintaining two slightly different model architectures.

- `prepare_flickr8k` loads and cleans captions and creates image-level train/validation/test splits.
- `create_sequences` makes caption-prefix / next-word pairs.
- `load_features` loads VGG16 vectors previously saved to disk.
- `CaptionBatches` loads only the required image vectors for each batch.
- `build_caption_model` constructs and compiles the actual CNN-feature + LSTM caption decoder.
- `train` connects all steps and saves results.

In [ ]:
from pathlib import Path
import sys
import numpy as np

# Works when opened at the repository root or inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import (
    clean_caption, CaptionTokenizer, create_sequences, prepare_flickr8k,
)
from src.feature_extraction import FEATURE_DIM, load_features
from src.train_lstm import (
    CaptionBatches, build_caption_model, prepare_batches, train,
)
print("Project root:", PROJECT_ROOT)
print("Expected VGG16 vector length:", FEATURE_DIM)

## 3. Preprocessing: turn captions into numerical sequences

A neural network cannot directly process the raw sentence `A dog runs`. It uses word IDs.

Your `data_preprocessing.py` code:
1. Lowercases and cleans the caption.
2. Adds `startseq` and `endseq` markers.
3. Fits a vocabulary using **training captions only**.
4. Converts captions into integer token sequences.
5. Generates input prefixes and next-word targets (teacher forcing).

The tokenizer's index **0 means padding**; index **1** represents an out-of-vocabulary word (`<unk>`). For a proper experiment, do **not** fit the tokenizer on test captions.

In [ ]:
# Small, self-contained demonstration; NOT the Flickr8k training data
example_captions = {
    "example.jpg": [clean_caption("A dog runs through grass.")]
}
tokenizer_example = CaptionTokenizer().fit(example_captions["example.jpg"])
ids, prefixes, targets = create_sequences(
    example_captions, tokenizer_example, max_length=9
)
print("Cleaned:", example_captions["example.jpg"][0])
print("Word IDs:", tokenizer_example.word_index)
print("Image IDs:", ids[:3])
print("Padded caption prefixes (first 3):\n", prefixes[:3])
print("Next-word targets (first 3):", [
    tokenizer_example.index_word[int(i)] for i in targets[:3]
])

### How next-word training examples are constructed

For `startseq a dog runs endseq`, the model learns these pairs:

| Caption-prefix input | Next-word target |
|---|---|
| `startseq` | `a` |
| `startseq a` | `dog` |
| `startseq a dog` | `runs` |
| `startseq a dog runs` | `endseq` |

This technique is called **teacher forcing**: during training, the LSTM receives the correct preceding words and learns to predict the next word.

**During caption generation**, those correct prefixes are unavailable. The model instead feeds back words generated in previous decoding steps. This difference explains why good next-word accuracy does not necessarily imply good full-caption BLEU.

## 4. Frozen VGG16 features: what does the LSTM receive?

`feature_extraction.py` runs each original image through ImageNet-pretrained VGG16, stopping at the **fc2** layer. That layer produces a **4096-number feature vector**.

Instead of loading the original image again for each caption prefix, the training code looks up a vector using the caption's image ID:

```python
features, row_of = load_features(...)
vector = features[row_of["example_image.jpg"]]
```

- `features` has shape **(number_of_images, 4096)**.
- `row_of` maps a filename to the correct feature row.
- The same image vector is reused for all five captions and their next-word examples.
- **Frozen** means that the CNN weights are not updated during LSTM training.

The next cell checks real saved features only **if they exist**; it does not run VGG16 or download images.

In [ ]:
features_dir = PROJECT_ROOT / "data" / "features"
if (features_dir / "features.npy").exists():
    real_features, row_of = load_features(features_dir, require_complete=True, mmap=True)
    print("Loaded saved VGG16 features:", real_features.shape)
    first_filename = next(iter(row_of))
    print("Sample filename:", first_filename)
    print("Feature vector shape:", real_features[row_of[first_filename]].shape)
else:
    print("Saved VGG16 features not found. This is fine for the demonstration.")
    print("For full training, run: python -m src.feature_extraction")

## 5. Why does `CaptionBatches` exist?

Flickr8k has multiple captions per image, and each caption is expanded into many training examples. Repeating the **4096-number** image vector across every word example would waste a large amount of memory.

`CaptionBatches(keras.utils.PyDataset)` stores:
- `features`: the saved image-feature matrix, ideally loaded with `mmap=True`.
- `rows`: the image-feature row for each next-word example.
- `prefixes`: padded caption tokens for each example.
- `targets`: the correct next word's integer ID.
- `order`: the indices to shuffle at each training epoch.

`__getitem__` gathers only one batch of feature vectors and caption prefixes and returns:

```python
({"image_features": float32(batch, 4096),
  "caption_input": int32(batch, max_length)},
 next_word_ids)
```

Training examples are shuffled; validation examples are not. The next cell uses a **tiny synthetic dataset** so you can inspect the shapes without downloading Flickr8k.

In [ ]:
# Three made-up images with 4096-dimensional features (not real Flickr8k)
rng = np.random.default_rng(42)
demo_features = rng.normal(size=(3, FEATURE_DIM)).astype("float32")

# Five next-word examples: each row points to one of the three image vectors
demo_rows = np.array([0, 0, 1, 2, 2], dtype="int32")
demo_prefixes = np.array([
    [0, 0, 0, 2, 3],
    [0, 0, 2, 3, 4],
    [0, 0, 0, 2, 5],
    [0, 0, 0, 2, 6],
    [0, 0, 2, 6, 7],
], dtype="int32")
demo_targets = np.array([4, 5, 6, 7, 8], dtype="int32")

demo_batches = CaptionBatches(
    demo_features, demo_rows, demo_prefixes, demo_targets,
    batch_size=2, shuffle=False,
)
batch_inputs, batch_targets = demo_batches[0]
print("Number of batches:", len(demo_batches))
print("Image batch shape:", batch_inputs["image_features"].shape)
print("Caption batch shape:", batch_inputs["caption_input"].shape)
print("Target batch shape:", batch_targets.shape)

## 6. Model architecture: explain each layer

The actual `build_caption_model()` function in `src/train_lstm.py` implements:

### Image branch
`Input(4096)` → `Dropout(0.4)` → `Dense(256, ReLU)`

The image is already a vector extracted by VGG16. `Dense(256)` projects the 4096 visual features into a 256-dimensional space.

### Text branch
`Input(max_length)` → `Embedding(vocab_size, 256, mask_zero=True)` → `Dropout(0.4)` → `LSTM(256)`

- **Embedding:** represents each word ID as a trainable 256-number vector.
- **Masking:** tells the recurrent layer which positions contain padding zeros.
- **LSTM:** summarises the preceding words with a 256-number representation, using its recurrent memory.

### Feature fusion and prediction
`Add([image_vector, text_vector])` → `Dense(256, ReLU)` → `Dropout` → `Dense(vocab_size, Softmax)`

The elementwise addition requires **matching vector lengths** (256 + 256); it is not a concatenation. Softmax produces a probability distribution over all vocabulary IDs. The next-word target is an integer index, not a one-hot vector.

**Note:** Your existing preprocessing uses **left-padding**. It works with masking, but right-padding can use a faster GPU-optimised LSTM kernel. Changing padding requires a deliberate change to preprocessing and corresponding inference code.

In [ ]:
# This small demo model is UNTRAINED, so its predictions are random-like.
demo_vocab_size = 16
demo_max_length = demo_prefixes.shape[1]

demo_model = build_caption_model(
    vocab_size=demo_vocab_size,
    max_length=demo_max_length,
    learning_rate=0.001,
    embedding_dim=256,
    lstm_units=256,
    dropout=0.4,
)
demo_model.summary()
print("Inputs:", demo_model.input_shape)
print("Output:", demo_model.output_shape)

# Each example receives one probability for every word in the vocabulary
probabilities = demo_model(batch_inputs, training=False).numpy()
print("Next-word probabilities shape:", probabilities.shape)
print("Probability sum for first example:", probabilities[0].sum())

## 7. Why Adam and sparse categorical cross-entropy?

In `build_caption_model`, compilation uses:

```python
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
```

**Adam** adjusts trainable weights from gradients and adapts the scale of parameter updates. The learning rate determines the initial step size.

**Sparse categorical cross-entropy** evaluates how much probability the model assigns to the correct next-word ID. Using integer targets avoids allocating a large one-hot vector for each training example.

**Sparse categorical accuracy** measures how often the most probable word matches the teacher-forced target. It is **not BLEU**, nor is it full-sentence caption accuracy.

**Dropout** temporarily zeroes some intermediate activations during training, which helps limit overfitting. It does not permanently remove neurons from the saved model.

## 8. Train versus validation versus test

The code separates **unique image IDs**, not caption rows:

| Split | How it is used |
|---|---|
| **Training (~80%)** | Fit vocabulary and update decoder weights |
| **Validation (~10%)** | Track generalisation, choose the best checkpoint, guide early stopping |
| **Test (~10%)** | Reserved for final greedy/beam BLEU comparison |

The same image may have five different reference captions, but all five remain in **one** split.

`prepare_batches(..., "train")` and `prepare_batches(..., "val")` create the supervised examples. The function explicitly rejects `split="test"`, preventing accidental use of test captions during fitting.

For published results, report the exact seed and split rule. This repository uses an 80/10/10 fixed-seed image split, which may differ from benchmark papers using the established Flickr8k split.

## 9. Training callbacks: why use them?

Your training implementation sets these Keras callbacks:

| Callback | Behaviour | Why it helps |
|---|---|---|
| `ModelCheckpoint` | Saves the model with the lowest `val_loss` | Keep the best validation checkpoint |
| `EarlyStopping` | Stops after a patience window with no improvement; restores best weights | Avoid unnecessary epochs and overfitting |
| `ReduceLROnPlateau` | Reduces learning rate when `val_loss` stops improving | Allow finer weight updates |
| `CSVLogger` | Logs loss/accuracy for each epoch | Preserve evidence for the report |

During `model.fit(train_batches, validation_data=val_batches, epochs=20, ...)`:
1. A training batch feeds saved VGG16 vectors plus correct caption prefixes.
2. The model predicts each next word.
3. Cross-entropy is computed, gradients update decoder parameters.
4. Validation loss is measured at the end of each epoch.
5. The callbacks decide whether to save, adjust learning rate or stop.

These are training methods, **not decoding strategies**. Greedy and beam search are applied after training.

In [ ]:
import keras

# Demonstration only: objects are created but no long training starts.
example_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
]
print("Callbacks:", [c.__class__.__name__ for c in example_callbacks])
print("Run the full notebook 03 or the optional full-training cell below to train.")

## 10. Run the full dataset training only when ready

The next cell uses the **real training pipeline** in `src/train_lstm.py` and saves trained model files. It is **disabled by default** to avoid accidentally launching multi-epoch training during a code walkthrough.

Before setting `RUN_FULL_TRAINING = True`:
- Install `requirements.txt`.
- Place `Images/` and `captions.txt` in `data/flickr8k/`.
- Finish `python -m src.feature_extraction` and verify `data/features/features.npy` exists.
- Prefer an environment with a compatible GPU (e.g. Google Colab or WSL2 with GPU support).

The full trainer runs validation but does not use the held-out test set.

In [ ]:
RUN_FULL_TRAINING = False  # Change to True only when data/features are present

if RUN_FULL_TRAINING:
    trained_model, training_result, prepared_data = train(
        data_dir=PROJECT_ROOT / "data" / "flickr8k",
        features_dir=PROJECT_ROOT / "data" / "features",
        output_dir=PROJECT_ROOT / "models" / "lstm",
        epochs=20,
        batch_size=128,
        learning_rate=0.001,
        dropout=0.4,
        patience=4,
        seed=42,
    )
else:
    print("Full training skipped. For complete training, run notebook 03.")

## 11. Understand the saved files and loss plots

When full training finishes, `models/lstm/` contains:

| File | Explanation |
|---|---|
| `best_caption_model.keras` | Best validation-loss checkpoint |
| `caption_model.keras` | Final model with restored best weights after early stopping |
| `tokenizer.json` | The mapping between words and integer IDs |
| `splits.json` | Exact image IDs in train/validation/test |
| `metadata.json` | Vocabulary size, max caption length and special tokens |
| `training_history.csv` | Per-epoch training and validation metrics |
| `history.json` | Same training history in JSON form |
| `training_config.json` | Hyperparameters and best validation loss |
| `training_loss.png` | Plot of training and validation cross-entropy |

**Interpretation tips:**
- Both losses decrease: the model is learning.
- Training loss decreases while validation loss rises: likely overfitting.
- Both losses remain high: consider insufficient training, inappropriate learning rate, features or architecture.
- Do **not** report a BLEU score from the loss curve; BLEU requires generating complete captions.

In [ ]:
import json
import matplotlib.pyplot as plt

output_dir = PROJECT_ROOT / "models" / "lstm"
history_file = output_dir / "history.json"
if history_file.exists():
    history = json.loads(history_file.read_text(encoding="utf-8"))
    plt.plot(history["loss"], label="Training loss")
    plt.plot(history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Sparse categorical cross-entropy")
    plt.title("Flickr8k LSTM training curves")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No history.json found. Train the real model first to generate curves.")

## 12. What happens after this training stage?

The pipeline in your image continues:

**Softmax next-word probabilities** → **greedy decoding or beam search** → **complete caption** → **compare to five human reference captions** → **BLEU scores**.

- **Greedy:** choose the highest-probability word each time.
- **Beam search:** retain multiple likely partial captions, e.g. beam width 3 or 5.
- **BLEU-1…BLEU-4:** evaluate generated caption word-sequence overlap with the references.
- Also compare generation time and review sample predictions for missing objects, incorrect actions, repetition and generic captions.

**Research report wording:**
> We used transfer learning with a frozen ImageNet-pretrained VGG16 encoder to extract 4096-dimensional image features. A Dense(256) layer transformed the visual vector, while a trainable Embedding(256) and LSTM(256) encoded the caption prefix. The image and text representations were combined through elementwise addition and projected through Dense(256) to a vocabulary softmax. We trained the decoder using teacher forcing, Adam optimisation and sparse categorical cross-entropy, with early stopping based on validation loss.

This describes the architecture and **intended training procedure**. Add actual results only after running the full model.

## 13. Quick reference: where to find the implementation

| Concept in this notebook | Source code |
|---|---|
| Text cleaning and tokenizer | `src/data_preprocessing.py`: `clean_caption`, `CaptionTokenizer` |
| Image-level split | `src/data_preprocessing.py`: `split_image_ids` |
| Next-word supervision | `src/data_preprocessing.py`: `create_sequences` |
| Frozen VGG16 features | `src/feature_extraction.py`: `extract_features`, `load_features` |
| Efficient batch loading | `src/train_lstm.py`: `CaptionBatches` |
| Image Dense / LSTM / Add / softmax | `src/train_lstm.py`: `build_caption_model` |
| Training and checkpointing | `src/train_lstm.py`: `train` |
| Full interactive trainer | `notebooks/03_lstm_training.ipynb` |

**Note:** The synthetic demonstration tests tensor shapes and explains operations; its output is not a trained captioning result.